# Final project: Tallybook's cloud review

Plan your final project, a cost, reliability and security review of a real-looking cloud estate, with every saving and risk measured and a 90-day plan the leadership can approve.

*From [CloudTech Academy](https://academy.cloudtechanalytics.com/learn/cloud-fundamentals-cost-reliability/final-project). Run the cells in order (Runtime → Run all), edit them, and experiment. Answer the practice questions on the lesson page.*

## The problem

Tallybook's CTO and finance director want one document: what the company should change in its cloud, in what order, what it will save, and what it will fix. They'll share it with investors, who have asked how Tallybook will keep cloud costs under control as it grows, and whether the month-end outages will happen again.

Your final project is that review, built from the data in this course.

## The concept

### The parts of the review

| Part | Built in |
| :-- | :-- |
| The bill explained: services, environments, teams, untagged spend, growth | lesson 2 |
| Savings: clean-up, rightsizing, schedules, pricing models | lessons 3 to 5 |
| Scaling for month-end | lesson 6 |
| Reliability: SLO, error budget, design changes | lesson 7 |
| Security findings and fixes | lesson 8 |
| Governance and unit costs | lesson 9 |

**Don't double-count savings.** Apply them in order: remove waste first, then rightsize what's left, then schedule, then commit to what remains. Each step's saving is calculated on what the previous step left.

## Example

A savings summary that applies the steps in order (monthly, in dollars, illustrative prices):

In [ ]:
import numpy as np
import pandas as pd

base = "https://academy.cloudtechanalytics.com/datasets/cloud/"
resources = pd.read_csv(base + "resources.csv", parse_dates=["created_date"])
util = pd.read_csv(base + "utilisation.csv")
traffic = pd.read_csv(base + "web_traffic.csv")
H = 730

vms = resources[(resources["type"] == "vm") & (resources["status"] == "running")].copy()
cpu = util.groupby("resource_id")["cpu_pct"].agg(p95=lambda x: x.quantile(0.95), peak="max")
vms = vms.join(cpu, on="resource_id")

idle = vms["peak"] < 5
steps = {"1. remove idle servers": (vms.loc[idle, "hourly_usd"] * H).sum()}
vms = vms[~idle]
api = vms["name"].str.startswith("prod-api")
steps["2. rightsize API servers (xlarge to large)"] = (api.sum() * (0.20 - 0.10)) * H
vms.loc[api, "hourly_usd"] = 0.10
dev = vms["name"].str.startswith("dev-")
steps["3. schedule development servers"] = (vms.loc[dev, "hourly_usd"] * (H - 11 * 21)).sum()
web_auto = np.maximum(2, np.ceil(traffic["requests"] / 5_400)).sum()
steps["4. autoscale web servers"] = (6 * 744 - web_auto) * 0.10 * H / 744
steps["5. commit 1 year: API servers and 2 web servers (35%)"] = (vms.loc[api, "hourly_usd"].sum() + 2 * 0.10) * H * 0.35

savings = pd.Series(steps).round(2)
print(savings)
print("Total monthly saving: $", round(savings.sum(), 2))

*Expected output:*

```
1. remove idle servers                                   292.00
2. rightsize API servers (xlarge to large)               292.00
3. schedule development servers                          399.20
4. autoscale web servers                                 189.86
5. commit 1 year: API servers and 2 web servers (35%)    153.30
dtype: float64
Total monthly saving: $ 1326.36
```

Add the clean-up of disks, IP addresses and snapshots from lesson 4 to complete the picture, and compare the total with the August bill.

## Walkthrough

1. Complete the savings table with storage clean-up, and express it as a share of the August bill and in naira (with your assumed rate).
2. Add the reliability changes and their costs: a multi-zone database costs roughly twice a single-zone one.
3. Rank the security fixes by risk.
4. Open the project brief on the course page and plan the write-up.

## Practice


**Dataset:** [resources.csv](https://academy.cloudtechanalytics.com/datasets/cloud/resources.csv), [billing.csv](https://academy.cloudtechanalytics.com/datasets/cloud/billing.csv), [utilisation.csv](https://academy.cloudtechanalytics.com/datasets/cloud/utilisation.csv), [web_traffic.csv](https://academy.cloudtechanalytics.com/datasets/cloud/web_traffic.csv), [outages.csv](https://academy.cloudtechanalytics.com/datasets/cloud/outages.csv), [access.csv](https://academy.cloudtechanalytics.com/datasets/cloud/access.csv)

**Practice:** What is the **total monthly saving** from the five steps, in dollars? Two decimal places.

*Work it out below, then enter your answer on the lesson page.*

In [ ]:
# Your code here

**Task:** Write the **executive summary** of your cloud review (100 to 200 words): the **monthly saving** and its share of the bill, the **reliability** changes and the SLO, the **security** priorities, and the **governance** that keeps it working, ending with a **90-day** plan.

*Work it out below, then enter your answer on the lesson page.*

In [ ]:
# Your code here

## Check your understanding